# TiendaTech — Pipeline analítico reproducible

Este cuaderno ejecuta y documenta las cinco transformaciones de D4 sobre las 600 000 órdenes analíticas. Los scripts `pipeline.py` y `baseline.py` son la fuente ejecutable para evitar divergencias entre el cuaderno y el experimento.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'spark' else Path.cwd()
SPARK_DIR = ROOT / 'spark'
OUTPUT = SPARK_DIR / 'out'
OUTPUT

## Cinco transformaciones

1. T1 temporal: fecha y trimestre.
2. T2 filtro: 2026, no canceladas, cantidades positivas y usuarios activos.
3. T3 joins: orden + detalle + usuario.
4. T4 agregación con ventana: top-10 trimestral y totales de cliente.
5. T5 ML: Bucketizer para segmentar gasto.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(SPARK_DIR / 'baseline.py'), '--overwrite'], cwd=ROOT, check=True)
subprocess.run(['powershell', '-ExecutionPolicy', 'Bypass', '-File', str(SPARK_DIR / 'ejecutar-pyspark.ps1'), '-Workers', '4', '-Salida', 'pyspark'], cwd=ROOT, check=True)

In [ ]:
subprocess.run([sys.executable, str(SPARK_DIR / 'validar_resultados.py')], cwd=ROOT, check=True)

In [ ]:
import pandas as pd

top = pd.read_parquet(OUTPUT / 'pyspark' / 'top_productos')
segmentos = pd.read_parquet(OUTPUT / 'pyspark' / 'segmentos_clientes')
display(top)
display(segmentos.groupby('segmento', as_index=False).agg(clientes=('usuario_id', 'count'), gasto_promedio=('gasto_total', 'mean')))

## Experimento completo

La siguiente celda tarda considerablemente: realiza diez repeticiones para `local[1]`, `local[2]`, `local[4]`, `local[8]` y pandas. Descarta la primera y última, calcula media, desviación, IC95 % y selecciona t pareada o Wilcoxon según Shapiro-Wilk.

In [ ]:
# Descomentar para generar la evidencia final.
# subprocess.run([sys.executable, str(SPARK_DIR / 'experimento.py'), '--repeticiones', '10', '--workers', '1', '2', '4', '8', '--incluir-pandas'], cwd=ROOT, check=True)